In [1]:
import pandas as pd

In [2]:
df=pd.read_csv(r'D:\Enterprise_ITSM_AI\data\raw\itsm_cleaned_dataset.csv')

Found that they are classes with less than 10 samples in assignment group,so removing them

In [7]:

group_counts = df["assignment_group"].value_counts()

rare_groups = group_counts[group_counts < 10].index

print("Rare Groups:")
print(rare_groups)


df = df[~df["assignment_group"].isin(rare_groups)]

print("New number of classes:", df["assignment_group"].nunique())

Rare Groups:
Index(['Group 79', 'Group 71', 'Group 18', 'Group 7', 'Group 8', 'Group 41',
       'Group 38', 'Group 11', 'Group 4', 'Group 16'],
      dtype='object', name='assignment_group')
New number of classes: 68


In [8]:
X = df[
    [
        "contact_type",
        "location",
        "u_symptom",
        "impact",
        "urgency",
        "knowledge",
        "notify",
        "opened_hour",
        "opened_day_of_week",
        "opened_month",
        "is_weekend"
    ]
]

y = df["assignment_group"]

In [9]:
y.value_counts()

assignment_group
Group 70    43474
Group 25     7679
Group 24     6752
Group 20     6170
Group 39     4728
            ...  
Group 2        45
Group 32       33
Group 78       22
Group 80       12
Group 36       11
Name: count, Length: 68, dtype: int64

checking the imbalance

In [10]:
y.value_counts().describe()

count       68.000000
mean      1874.352941
std       5380.191218
min         11.000000
25%        165.500000
50%        565.500000
75%       1929.000000
max      43474.000000
Name: count, dtype: float64

In [11]:
print("Number of classes:", y.nunique())
print("Classes with <5 samples:", (y.value_counts() < 5).sum())
print("Classes with <10 samples:", (y.value_counts() < 10).sum())
print("Classes with <20 samples:", (y.value_counts() < 20).sum())

Number of classes: 68
Classes with <5 samples: 0
Classes with <10 samples: 0
Classes with <20 samples: 2


removing the classes which has less than 10 samples from the dataset to reduce the imbalance and make the fair training

After removing classes with less than 10 samples:Number of classes: 68
Classes with <5 samples: 0
Classes with <10 samples: 0
Classes with <20 samples: 2

Label Encoding

In [12]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y_encoded=le.fit_transform(y)

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [16]:
import xgboost
from xgboost import XGBClassifier

In [17]:
preprocessing_data=ColumnTransformer(
    transformers=[
        (
            'trf1',
            OrdinalEncoder(categories=[['1 - High','2 - Medium','3 - Low'],['1 - High','2 - Medium','3 - Low']]),
            ['impact','urgency']
        ),
        (
            'trf2',
            OneHotEncoder(handle_unknown='ignore'),
            ['contact_type','location','u_symptom','notify','opened_day_of_week']
        )],
    remainder='passthrough')
#creating multiple pipeline for different models
dt_pipeline=Pipeline([
    ('preprocessor',preprocessing_data),
    ('model',DecisionTreeClassifier(max_depth=3,min_samples_leaf=20,min_samples_split=10,random_state=42))
])

rf_pipeline=Pipeline([
    ('preprocessor',preprocessing_data),
    ('model',RandomForestClassifier(n_estimators=300,  max_depth=15,min_samples_split=5,min_samples_leaf=2,max_features='sqrt',random_state=42,n_jobs=-1))
])

xgboost_pipeline=Pipeline([
    ('preprocessor',preprocessing_data),
    ('model',XGBClassifier(random_state=42))
])



In [18]:
dt_pipeline.fit(X_train,y_train)
rf_pipeline.fit(X_train,y_train)
xgboost_pipeline.fit(X_train,y_train)

c:\Users\bharg\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('trf1',
                                                  OrdinalEncoder(categories=[['1 '
                                                                              '- '
                                                                              'High',
                                                                              '2 '
                                                                              '- '
                                                                              'Medium',
                                                                              '3 '
                                                                              '- '
                                                                              'Low'],
                                                                             ['1 '
                                                                              '- '
                                                                              'High',
                                                                              '2 '
                                                                              '- '
                                                                              'Medium',
                                                                              '3 '
                                                                              '- '
                                                                              'Low']]),
                                                  ['impact', 'urgency']),
                                                 ('trf2',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['contact_type', 'location',
                                                   'u_symptom', 'notify',
                                                   'opened_day_of_week'])])...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [19]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [20]:
results={}
def evaluate_model(name,model,X_test,y_test):
    y_pred=model.predict(X_test)
    metrics={
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro"),
        "Recall": recall_score(y_test, y_pred, average="macro"),
        "F1 Score": f1_score(y_test, y_pred, average="macro")
    }
    results[name]=metrics

In [21]:
evaluate_model("Decision Tree", dt_pipeline, X_test, y_test)
evaluate_model("Random Forest", rf_pipeline, X_test, y_test)
evaluate_model("XGBoost", xgboost_pipeline, X_test, y_test)

c:\Users\bharg\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\bharg\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\bharg\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
results_df=pd.DataFrame(results).T
results_df = results_df.round(4)
print(results_df)

               Accuracy  Precision  Recall  F1 Score
Decision Tree    0.3069     0.0044  0.0145    0.0068
Random Forest    0.3278     0.2232  0.0349    0.0368
XGBoost          0.4742     0.6053  0.3544    0.4233


In [32]:
import joblib
import os

MODEL_DIR = r"D:\Enterprise_ITSM_AI\models_saved"

# Save Assignment Group XGBoost pipeline
joblib.dump(
    xgboost_pipeline,
    os.path.join(MODEL_DIR, "assignment_group_prediction_pipeline.joblib")
)

# Save Assignment Group LabelEncoder
joblib.dump(
    le,
    os.path.join(MODEL_DIR, "assignment_group_label_encoder.joblib")
)


['D:\\Enterprise_ITSM_AI\\models_saved\\assignment_group_label_encoder.joblib']

In [33]:
import os
import joblib

MODEL_DIR = r"D:\Enterprise_ITSM_AI\models_saved"

assignment_model = joblib.load(
    os.path.join(
        MODEL_DIR,
        "assignment_group_prediction_pipeline.joblib"
    )
)

assignment_encoder = joblib.load(
    os.path.join(
        MODEL_DIR,
        "assignment_group_label_encoder.joblib"
    )
)

print("Assignment Group model loaded successfully!")
print("Number of assignment groups:", len(assignment_encoder.classes_))

Assignment Group model loaded successfully!
Number of assignment groups: 69


In [34]:
sample = X_test.iloc[[0]]

encoded_prediction = assignment_model.predict(sample)[0]

predicted_group = assignment_encoder.inverse_transform(
    [encoded_prediction]
)[0]

print("Predicted Assignment Group:", predicted_group)

Predicted Assignment Group: Group 24


consider baseline model as XGBoost
Tuning it for better accuracy

In [23]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [4, 6, 8, 10],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
    "model__min_child_weight": [1, 3, 5],
    "model__gamma": [0, 0.1, 0.3]
}

In [24]:
random_search = RandomizedSearchCV(
    estimator=xgboost_pipeline,
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

In [26]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


c:\Users\bharg\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('trf1',
                                                                               OrdinalEncoder(categories=[['1 '
                                                                                                           '- '
                                                                                                           'High',
                                                                                                           '2 '
                                                                                                           '- '
                                                                                                           'Medium',
                                                                                                           '3 '
                                                                                                           '- '
                                                                                                           'Low'],
                                                                                                          ['1 '
                                                                                                           '- '
                                                                                                           'High',
                                                                                                           '2 '
                                                                                                           '- '
                                                                                                           'Medium',
                                                                                                           '3 '
                                                                                                           '- '
                                                                                                           'Low']]),
                                                                               ['impact',
                                                                                'urgency']),
                                                                              ('trf2',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               ['contact_type',
                                                                                'location',
                                                                                'u_symptom'...
                                                            num_parallel_tree=None, ...))]),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'model__colsample_bytree': [0.7, 0.8,
                                                                    1.0],
                                        'model__gamma': [0, 0.1, 0.3],
                                        'model__learning_rate': [0.01, 0.05,
                                                                 0.1],
                                        'model__max_depth': [4, 6, 8, 10],
                                        'model__min_child_weight': [1, 3, 5],
                                        'model__n_estimators': [100, 200, 300],
                                        'model__subsample': [0.7, 0.8, 1.0]},
                   random_state=42, scoring='f1_macro', verbose=2)

In [27]:
print(random_search.best_params_)

{'model__subsample': 0.8, 'model__n_estimators': 300, 'model__min_child_weight': 1, 'model__max_depth': 6, 'model__learning_rate': 0.05, 'model__gamma': 0.1, 'model__colsample_bytree': 0.7}


In [28]:
print(random_search.best_score_)

0.37052245907125547


In [29]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="macro"))
print("Recall   :", recall_score(y_test, y_pred, average="macro"))
print("F1 Score :", f1_score(y_test, y_pred, average="macro"))

Accuracy : 0.462024422954754
Precision: 0.6150416477723838
Recall   : 0.32126347234521674
F1 Score : 0.3964465995427087


c:\Users\bharg\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [35]:
resolver_mapping = (
    df.dropna(subset=["assignment_group", "resolved_by"])
      .groupby(["assignment_group", "resolved_by"])
      .size()
      .reset_index(name="incident_count")
      .sort_values(
          ["assignment_group", "incident_count"],
          ascending=[True, False]
      )
)

print(resolver_mapping.head(20))

   assignment_group      resolved_by  incident_count
21         Group 10  Resolved by 182             870
39         Group 10   Resolved by 35             426
5          Group 10  Resolved by 125             400
20         Group 10  Resolved by 177              49
3          Group 10  Resolved by 119              24
24         Group 10  Resolved by 188              17
34         Group 10  Resolved by 226              16
45         Group 10   Resolved by 69              14
14         Group 10  Resolved by 159              13
9          Group 10  Resolved by 139              12
17         Group 10  Resolved by 166              12
31         Group 10  Resolved by 213              12
43         Group 10   Resolved by 62              11
10         Group 10  Resolved by 140               9
13         Group 10  Resolved by 158               9
29         Group 10   Resolved by 20               9
1          Group 10   Resolved by 11               7
46         Group 10   Resolved by 70          